In [2]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re
from joblib import Parallel, delayed
from pathlib import Path
import matplotlib.pyplot as plt
import alphashape
from itertools import combinations
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import ProcessPoolExecutor, as_completed


from shapely import points, contains
import random

In [1]:
#%pip install -e ..

In [3]:
from proj2dhullsampler import HistoryMatching

In [4]:
working_dir = '/glade/work/qingyuany/camml_re'
case_name = "test"

para = xr.open_dataset("/glade/work/qingyuany/camml_re/v0/parameter_34_100.nc")
para = para.to_dataframe().drop(columns = 'Sample_nmb')
para.index = para.index +1




In [5]:
test_case = HistoryMatching(working_dir, case_name, para, threshold_level=2.0)
test_case.drop_by_name(["TGCLDLWP", "CLDTOT"])
test_case.drop_by_n_survive(n_survive = 200000)
test_case.update_meta()



In [6]:
test_case.group_para_climatology(overlapping_threshold = 10000)
summary2d = test_case.shuffle_vars()

micro_mg_vtrmi_factor                    and micro_mg_dcs                            :        0
micro_mg_dcs                             and microp_aero_wsub_scale                  :   255301
micro_mg_dcs                             and microp_aero_wsubi_scale                 :   615788
micro_mg_vtrmi_factor                    and microp_aero_wsub_scale                  :   673309
microp_aero_wsub_scale                   and microp_aero_npccn_scale                 :     1711
micro_mg_vtrmi_factor                    and zmconv_capelmt                          :   219088
micro_mg_vtrmi_factor                    and clubb_c2rt                              :   353025
micro_mg_vtrmi_factor                    and zmconv_tiedke_add                       :   635723
micro_mg_vtrmi_factor                    and clubb_c14                               :   918002
micro_mg_vtrmi_factor                    and micro_mg_berg_eff_factor                :   954932
micro_mg_dcs                            

In [7]:
list(summary2d.values())[1]

,var1,var2,count
3,FSNTOA_zonal_70to75,FSNTOA_zonal_-70to-65,4509
9,FSNTOA_zonal_-75to-70,FSNTOA_zonal_-70to-65,36687
0,FSNTOA_zonal_70to75,FSNTOA_zonal_-65to-60,156042
1,FSNTOA_zonal_70to75,SWCF_zonal_-65to-60,176736
8,SWCF_zonal_-65to-60,FSNTOA_zonal_-70to-65,293257
6,FSNTOA_zonal_-65to-60,FSNTOA_zonal_-70to-65,293257
2,FSNTOA_zonal_70to75,FSNTOA_zonal_-75to-70,369007
5,FSNTOA_zonal_-65to-60,FSNTOA_zonal_-75to-70,485110
7,SWCF_zonal_-65to-60,FSNTOA_zonal_-75to-70,505991
4,FSNTOA_zonal_-65to-60,SWCF_zonal_-65to-60,777412


In [8]:
no_overlap_2d_vars = ['LWCF_zonal_-75to-70', 'FSNTOA_zonal_-70to-65']

In [9]:
test_case.drop_no_overlap2d_vars(no_overlap_2d_vars)
test_case.group_para_climatology()
summary2d = test_case.shuffle_vars()
summary2d

micro_mg_vtrmi_factor                    and micro_mg_dcs                            :    77151
micro_mg_dcs                             and microp_aero_wsub_scale                  :   255301
micro_mg_dcs                             and microp_aero_wsubi_scale                 :   615788
micro_mg_vtrmi_factor                    and microp_aero_wsub_scale                  :   673309
microp_aero_wsub_scale                   and microp_aero_npccn_scale                 :   148048
micro_mg_vtrmi_factor                    and zmconv_capelmt                          :   219088
micro_mg_vtrmi_factor                    and clubb_c2rt                              :   353025
micro_mg_vtrmi_factor                    and zmconv_tiedke_add                       :   635723
micro_mg_vtrmi_factor                    and clubb_c14                               :   918002
micro_mg_vtrmi_factor                    and micro_mg_berg_eff_factor                :   954932
micro_mg_dcs                            

{}

In [10]:
test_case.paras_vars

{('micro_mg_vtrmi_factor', 'micro_mg_dcs'): ['FLUT_zonal_45to50',
  'TMQ_zonal_-5to0',
  'PRECT_zonal_40to45',
  'FSNTOA_zonal_20to25',
  'LWCF_zonal_-70to-65',
  'FLUT_zonal_50to55',
  'FSNTOA_zonal_25to30',
  'FLUT_zonal_65to70',
  'PRECT_zonal_-15to-10',
  'LWCF_zonal_0to5',
  'LWCF_zonal_-35to-30',
  'PRECT_zonal_-45to-40',
  'FSNTOA_zonal_15to20',
  'SWCF_zonal_-25to-20',
  'LWCF_zonal_-15to-10',
  'LWCF_zonal_50to55',
  'LWCF_zonal_30to35',
  'LWCF_zonal_-45to-40',
  'PRECT_zonal_-10to-5',
  'LWCF_zonal_40to45',
  'FLUT_zonal_5to10',
  'LWCF_zonal_-5to0',
  'SWCF_zonal_-20to-15',
  'FLUT_zonal_-10to-5',
  'LWCF_zonal_-60to-55',
  'TMQ_zonal_-25to-20',
  'FLUT_zonal_-60to-55',
  'FLUT_zonal_0to5',
  'TMQ_zonal_15to20',
  'FLUT_zonal_-5to0',
  'FSNTOA_zonal_-25to-20',
  'FLUT_zonal_-75to-70',
  'TMQ_zonal_30to35',
  'FLUT_zonal_30to35',
  'TMQ_zonal_-10to-5',
  'FLUT_zonal_-55to-50',
  'TMQ_zonal_0to5',
  'PRECT_zonal_50to55',
  'TMQ_zonal_20to25',
  'FLUT_zonal_25to30',
  'FLUT_zo

In [11]:
test_case.drop_by_nvar_per_pair(n_var_thre=1)

In [12]:
test_case.paras_vars

{('micro_mg_vtrmi_factor', 'micro_mg_dcs'): ['FLUT_zonal_45to50',
  'TMQ_zonal_-5to0',
  'PRECT_zonal_40to45',
  'FSNTOA_zonal_20to25',
  'LWCF_zonal_-70to-65',
  'FLUT_zonal_50to55',
  'FSNTOA_zonal_25to30',
  'FLUT_zonal_65to70',
  'PRECT_zonal_-15to-10',
  'LWCF_zonal_0to5',
  'LWCF_zonal_-35to-30',
  'PRECT_zonal_-45to-40',
  'FSNTOA_zonal_15to20',
  'SWCF_zonal_-25to-20',
  'LWCF_zonal_-15to-10',
  'LWCF_zonal_50to55',
  'LWCF_zonal_30to35',
  'LWCF_zonal_-45to-40',
  'PRECT_zonal_-10to-5',
  'LWCF_zonal_40to45',
  'FLUT_zonal_5to10',
  'LWCF_zonal_-5to0',
  'SWCF_zonal_-20to-15',
  'FLUT_zonal_-10to-5',
  'LWCF_zonal_-60to-55',
  'TMQ_zonal_-25to-20',
  'FLUT_zonal_-60to-55',
  'FLUT_zonal_0to5',
  'TMQ_zonal_15to20',
  'FLUT_zonal_-5to0',
  'FSNTOA_zonal_-25to-20',
  'FLUT_zonal_-75to-70',
  'TMQ_zonal_30to35',
  'FLUT_zonal_30to35',
  'TMQ_zonal_-10to-5',
  'FLUT_zonal_-55to-50',
  'TMQ_zonal_0to5',
  'PRECT_zonal_50to55',
  'TMQ_zonal_20to25',
  'FLUT_zonal_25to30',
  'FLUT_zo

In [13]:
test_case.build_hulls()

In [14]:
test_case.orchestrate()

Running ('micro_mg_vtrmi_factor', 'micro_mg_dcs'), the 0th simulation
There is overlap for ('micro_mg_vtrmi_factor', 'micro_mg_dcs'). Proceed to the next parameter pair
Running ('micro_mg_dcs', 'microp_aero_wsub_scale'), the 1th simulation
There is overlap for ('micro_mg_dcs', 'microp_aero_wsub_scale'). Proceed to the next parameter pair
Running ('micro_mg_dcs', 'microp_aero_wsubi_scale'), the 2th simulation
There is overlap for ('micro_mg_dcs', 'microp_aero_wsubi_scale'). Proceed to the next parameter pair
Running ('micro_mg_vtrmi_factor', 'microp_aero_wsub_scale'), the 3th simulation
There is overlap for ('micro_mg_vtrmi_factor', 'microp_aero_wsub_scale'). Proceed to the next parameter pair
Running ('microp_aero_wsub_scale', 'microp_aero_npccn_scale'), the 4th simulation
There is overlap for ('microp_aero_wsub_scale', 'microp_aero_npccn_scale'). Proceed to the next parameter pair
Running ('micro_mg_vtrmi_factor', 'zmconv_capelmt'), the 5th simulation
There is overlap for ('micro_mg_v

In [15]:
test_case.draw(n_pts=50000, n_threshold=5000, sample_threshold=10**6, max_workers=8, n_max=1000)

In [16]:
test_case.results.realscale_samples

,micro_mg_max_nicons,micro_mg_vtrmi_factor,micro_mg_iaccr_factor,micro_mg_berg_eff_factor,micro_mg_accre_enhan_fact,micro_mg_homog_size,micro_mg_dcs,clubb_c1,clubb_C8,clubb_c11,...,zmconv_capelmt,seasalt_emis_scale,dust_emis_fact,sol_factb_interstitial,sol_factic_interstitial,microp_aero_wsub_min,microp_aero_wsubi_min,microp_aero_wsub_scale,microp_aero_wsubi_scale,microp_aero_npccn_scale
0,5.085518e+09,3.929334,0.554600,0.574967,5.581729,0.000115,0.000408,0.687813,3.538690,0.760126,...,151.697158,1.101171,0.535446,0.719078,0.205807,0.179732,0.068041,1.938315,2.114598,0.734393
1,2.910738e+09,4.587804,0.524882,0.871795,2.203972,0.000170,0.000366,1.420181,1.099362,0.754723,...,66.829103,2.134961,0.111655,0.997026,0.300330,0.189593,0.148207,1.288124,2.743740,2.567392
2,8.435482e+08,4.872675,0.321374,0.816359,1.619403,0.000013,0.000315,2.954336,4.793016,0.400098,...,135.174083,2.398829,0.147599,0.120803,0.939548,0.495290,0.174523,1.524380,2.856169,2.075626
3,4.944764e+09,3.106683,0.458829,0.860387,8.436325,0.000198,0.000340,0.482575,1.736848,0.330456,...,94.633055,0.899453,0.450598,0.683760,0.879237,0.236894,0.097357,1.074611,0.816518,1.393446
4,1.442921e+09,4.741596,0.575197,0.327953,3.237176,0.000084,0.000440,2.534740,4.830674,0.395294,...,80.715191,0.752088,0.735733,0.142311,0.807701,0.085235,0.187971,1.412915,1.643700,2.739248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,9.895705e+09,4.587014,0.345735,0.508833,0.429077,0.000063,0.000533,0.721955,2.786986,0.321454,...,128.668990,1.995702,0.706025,0.671388,0.560022,0.147824,0.021455,1.366969,0.431769,2.440874
942,4.638879e+09,4.680991,0.571665,0.464321,3.781759,0.000073,0.000355,0.754171,2.745752,0.636984,...,85.539149,1.365807,0.597398,0.736789,0.954071,0.248569,0.053716,1.418162,2.971545,1.817906
943,7.692298e+09,4.545168,0.876568,0.493333,2.641149,0.000133,0.000590,1.077543,1.217596,0.704410,...,40.098753,1.223045,0.787474,0.316162,0.757148,0.110441,0.091608,1.099264,1.902816,1.246269
944,2.192113e+09,4.755167,0.727224,0.797386,1.807451,0.000178,0.000508,0.780982,4.168354,0.606373,...,115.449873,0.905721,0.694295,0.236212,0.204975,0.092672,0.067398,0.939348,1.841113,1.560875


In [17]:

#plot_histograms_grid_5([para, test_case.results.realscale_samples])

In [18]:
test_case.save_samples(n = 100)

(946, 34)
['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018', '019', '020', '021', '022', '023', '024', '025', '026', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '061', '062', '063', '064', '065', '066', '067', '068', '069', '070', '071', '072', '073', '074', '075', '076', '077', '078', '079', '080', '081', '082', '083', '084', '085', '086', '087', '088', '089', '090', '091', '092', '093', '094', '095', '096', '097', '098', '099', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '1

In [19]:
test_case.write_specifications()